# Four-Class Frame CNN Classifier: Fine-Tuned ResNet50

This notebook trains a four-class frame classifier from the checked dataset in `stoppage_detection_and_classification/cnn_classifier/data/training_frames`.

The model classes are:

- `no_bottle`
- `normal`
- `fallen_before_entry`
- `fallen_in_view`

The training pipeline uses:

- ImageNet-pretrained ResNet50 with a custom four-class head.
- Head-only training followed by fine-tuning of `layer3` and `layer4`.
- Aspect-preserving letterbox preprocessing for training and inference.
- Weighted sampling with an unweighted cross-entropy loss.
- Best-checkpoint selection by validation clip-level macro F1.

Training, validation and test inputs come from `training_frames_50_manifest.csv`; videos are not decoded to build the training dataset. Every retained source frame contributes one original and one ORB-transformed image. Complete video references and byte-identical linked references remain in one split, while dates are ignored.

The CNN itself trains on individual frames. Clip grouping is used only for leakage-safe splitting, validation checkpoint selection, held-out reporting, and optional temporal deployment inference.


## Dataset and training contract

The authoritative dataset is `stoppage_detection_and_classification/cnn_classifier/data/training_frames`: each retained source image has one `original` and one `orb_transformed` model input, recorded in `training_frames_50_manifest.csv`.

Before training, the notebook verifies that:

- Every manifest image exists inside the retained dataset folder.
- Every source image has exactly one original and one ORB-transformed row.
- A complete video reference remains in one train/validation/test split.
- Byte-identical images cannot cross splits.
- Validation and test contain equal video counts from every class.


---

## Chapter 1: Setup and configuration

In [ ]:
import hashlib
import json
import os
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.v2 as T
from IPython.display import Image as DisplayImage, display
from PIL import Image as PILImage, ImageOps
from sklearn.metrics import (
    auc,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

# Find the repository root whether Jupyter starts in the repo or a pipeline folder.
working_directory = Path.cwd().resolve()
repository_candidates = [working_directory, *working_directory.parents]
REPO_ROOT = next(
    candidate
    for candidate in repository_candidates
    if (candidate / "VideoModule").is_dir()
)
os.chdir(REPO_ROOT)
PIPELINE_ROOT = REPO_ROOT / "stoppage_detection_and_classification"
CNN_CLASSIFIER_DIR = PIPELINE_ROOT / "cnn_classifier"
TRAINING_DATA_DIR = CNN_CLASSIFIER_DIR / "data" / "training_frames"
DEMO_DIR = REPO_ROOT / "demo_notebooks" / "AnomalyClassification"
for import_path in (REPO_ROOT, CNN_CLASSIFIER_DIR, DEMO_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import _shared as S
from VideoModule.io.decode import decode_clip_rgb_preserve_aspect_with_fps
from VideoModule.plotting.confusion_matrix_plot import render_confusion_matrix

plt.rcParams["figure.figsize"] = (16, 6)
plt.rcParams["figure.dpi"] = 100


@dataclass(frozen=True)
class Config:
    """All tuneable parameters in one place."""

    # The retained 50/50 manifest is the only CNN dataset input.
    frame_dataset_root: Path = TRAINING_DATA_DIR
    frame_manifest_path: Path = (
        TRAINING_DATA_DIR
        / "training_frames_50_manifest.csv"
    )
    output_dir: Path = CNN_CLASSIFIER_DIR / "output" / "02_train_cnn"
    model_save_path: Path = CNN_CLASSIFIER_DIR / "output" / "02_train_cnn" / "best_model.pth"

    # Live-video inference. None processes every frame; use 10.0 for 10 FPS.
    inference_sample_fps: float | None = 60.0
    rolling_window_seconds: float = 0.5
    persistent_seconds: float = 0.3
    min_falling_seconds: float = 0.10
    peak_support_seconds: float = 0.10
    normal_clear_seconds: float = 0.5
    fallen_clear_seconds: float = 0.75
    normal_threshold: float = 0.75
    fault_activity_threshold: float = 0.50
    falling_threshold: float = 0.60
    falling_peak_threshold: float = 0.80
    fallen_threshold: float = 0.65

    # Video-level split. Validation and test use equal video counts per class.
    train_ratio: float = 0.40
    val_ratio: float = 0.20
    test_ratio: float = 0.40

    # Image and augmentation
    img_size: int = 224

    # Training hyperparameters
    batch_size: int = 64
    epochs_head: int = 7
    epochs_finetune: int = 13
    lr_head: float = 1e-3
    lr_finetune: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 5
    num_workers: int = 0  # Keep zero in Windows notebooks.

    # Device and reproducibility
    device: str = "cuda:0"
    seed: int = 42


CFG = Config()

CLASS_FOLDERS = {
    "no_bottle": 0,
    "normal": 1,
    "fallen_before_entry": 2,
    "fallen_in_view": 3,
}
CLASS_NAMES = ["no_bottle", "normal", "fallen_before_entry", "fallen_in_view"]
ROTATE_FROM_TIMESTAMP_UTC = pd.Timestamp("2026-06-21 00:00:00")
VIDEO_TIMESTAMP_PATTERN = re.compile(
    r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d{6})"
)

# Validate configuration before reading the manifest.
assert abs(CFG.train_ratio + CFG.val_ratio + CFG.test_ratio - 1.0) < 1e-9
if CFG.inference_sample_fps is not None and CFG.inference_sample_fps <= 0:
    raise ValueError("inference_sample_fps must be None or positive")
if CFG.rolling_window_seconds <= 0:
    raise ValueError("rolling_window_seconds must be positive")
for threshold_name in (
    "normal_threshold",
    "fault_activity_threshold",
    "falling_threshold",
    "falling_peak_threshold",
    "fallen_threshold",
):
    threshold = getattr(CFG, threshold_name)
    if not 0.0 <= threshold <= 1.0:
        raise ValueError(f"{threshold_name} must be between 0 and 1")

CFG.output_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TORCH_HOME", str(CNN_CLASSIFIER_DIR / "output" / "model_cache" / "torch"))

DEVICE = torch.device(CFG.device if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

S.print_hardware_banner()
print(f"Project root:       {REPO_ROOT}")
print(f"Device:             {DEVICE}")
print(f"50/50 frame data:   {CFG.frame_dataset_root}")
print(f"Training manifest:  {CFG.frame_manifest_path}")
inference_rate = "every frame" if CFG.inference_sample_fps is None else f"{CFG.inference_sample_fps:g} FPS"
print(f"Live inference:     {inference_rate}, {CFG.rolling_window_seconds:g}s rolling window")
print(f"Train/Val/Test:     {CFG.train_ratio:.0%} / {CFG.val_ratio:.0%} / {CFG.test_ratio:.0%}")
print(f"Output dir:         {CFG.output_dir}")


---

## Chapter 2: Load and validate the frame manifest

Each video reference is one split unit, and video references sharing byte-identical frames are linked into the same split. Dates are ignored, while every source frame and its original/ORB versions remain with their video.


In [ ]:
def path_is_within(path, expected_root):
    """Return True when a resolved path is inside the expected root."""
    try:
        path.relative_to(expected_root)
        return True
    except ValueError:
        return False


def filesystem_path(path):
    """Return an extended Windows path when the absolute path may exceed MAX_PATH."""
    resolved_path = Path(path).resolve()
    if os.name == "nt" and not str(resolved_path).startswith("\\\\?\\"):
        return Path(f"\\\\?\\{resolved_path}")
    return resolved_path


# Load the self-contained frame manifest instead of decoding videos.
if not CFG.frame_dataset_root.is_dir():
    raise FileNotFoundError(f"Missing 50/50 frame folder: {CFG.frame_dataset_root}")
if not CFG.frame_manifest_path.is_file():
    raise FileNotFoundError(f"Missing frame manifest: {CFG.frame_manifest_path}")

frame_manifest_df = pd.read_csv(CFG.frame_manifest_path)
required_manifest_columns = {
    "class_name",
    "clip_id",
    "split_group",
    "video_path",
    "video_name",
    "video_timestamp",
    "image_kind",
    "source_image_path",
    "output_image_path",
    "is_hard_negative",
}
missing_manifest_columns = required_manifest_columns - set(frame_manifest_df.columns)
if missing_manifest_columns:
    raise ValueError(f"Frame manifest is missing columns: {sorted(missing_manifest_columns)}")

frame_manifest_df = frame_manifest_df[
    frame_manifest_df["class_name"].isin(CLASS_FOLDERS)
].copy()
if frame_manifest_df.empty:
    raise ValueError("The frame manifest contains no configured classes")

# Resolve manifest paths through the extended Windows path form when needed.
dataset_root_resolved = filesystem_path(CFG.frame_dataset_root)
for path_column in ("source_image_path", "output_image_path"):
    frame_manifest_df[path_column] = frame_manifest_df[path_column].map(
        lambda value: str(filesystem_path(value))
    )

invalid_dataset_paths = []
missing_dataset_paths = []
for path_column in ("source_image_path", "output_image_path"):
    for image_path in frame_manifest_df[path_column].unique():
        resolved_path = Path(image_path)
        if not path_is_within(resolved_path, dataset_root_resolved):
            invalid_dataset_paths.append(image_path)
        elif not resolved_path.is_file():
            missing_dataset_paths.append(image_path)

if invalid_dataset_paths:
    raise ValueError(
        "Manifest image paths outside stoppage_detection_and_classification/cnn_classifier/data/training_frames: "
        f"{invalid_dataset_paths[:3]}"
    )
if missing_dataset_paths:
    raise FileNotFoundError(
        f"Missing retained dataset images: {missing_dataset_paths[:3]}"
    )
if frame_manifest_df["output_image_path"].duplicated().any():
    raise ValueError("The frame manifest contains duplicate output image paths")

# Require exactly one original and one ORB row for every checked source frame.
expected_image_kinds = {"original", "orb_transformed"}
invalid_pairs = []
for pair_key, pair_rows in frame_manifest_df.groupby(
    ["clip_id", "source_image_path"],
    sort=False,
):
    image_kinds = set(pair_rows["image_kind"])
    if len(pair_rows) != 2 or image_kinds != expected_image_kinds:
        invalid_pairs.append((pair_key, len(pair_rows), sorted(image_kinds)))
if invalid_pairs:
    raise ValueError(
        "Every source frame must have one original and one ORB copy: "
        f"{invalid_pairs[:3]}"
    )

# Build one clip record from the manifest for labels and provenance.
clip_inventory = []
for clip_id, clip_rows in frame_manifest_df.groupby("clip_id", sort=True):
    class_values = clip_rows["class_name"].dropna().unique()
    video_path_values = clip_rows["video_path"].dropna().unique()
    timestamp_values = clip_rows["video_timestamp"].dropna().unique()
    split_group_values = clip_rows["split_group"].dropna().unique()
    if len(class_values) != 1 or len(video_path_values) != 1:
        raise ValueError(f"Conflicting clip metadata for: {clip_id}")
    if len(timestamp_values) != 1 or len(split_group_values) != 1:
        raise ValueError(f"Incomplete clip metadata for: {clip_id}")

    class_name = str(class_values[0])
    # Keep the video path as a grouping identifier; the source video is not required.
    video_path = Path(str(video_path_values[0]))

    kind_counts = clip_rows["image_kind"].value_counts()
    original_count = int(kind_counts.get("original", 0))
    orb_count = int(kind_counts.get("orb_transformed", 0))
    if original_count != orb_count:
        raise ValueError(f"Clip is not 50/50 original/ORB: {clip_id}")

    clip_inventory.append({
        "clip_id": str(clip_id),
        "manifest_split_group": str(split_group_values[0]),
        "folder_name": class_name,
        "class_code": CLASS_FOLDERS[class_name],
        "class_name": CLASS_NAMES[CLASS_FOLDERS[class_name]],
        "video_path": video_path,
        "video_timestamp": pd.to_datetime(timestamp_values[0]),
        "original_frame_count": original_count,
        "orb_frame_count": orb_count,
    })

if not clip_inventory:
    raise ValueError("The frame manifest produced no clip records")

# Report clip provenance without using it as a split boundary.
clip_counts = Counter(record["class_code"] for record in clip_inventory)
print(f"Total clips retained:           {len(clip_inventory)}")
print(f"Manifest frame rows:            {len(frame_manifest_df):,}")
print("\nClass totals:")
for class_code, class_name in enumerate(CLASS_NAMES):
    print(f"  {class_name:<20s} {clip_counts[class_code]:>3} clips")

# Build one indivisible record for every original/ORB source-frame pair.
pair_records = []
for pair_key, pair_rows in frame_manifest_df.groupby(
    ["clip_id", "source_image_path"],
    sort=True,
):
    clip_id, source_image_path = pair_key
    class_values = pair_rows["class_name"].dropna().unique()
    if len(class_values) != 1:
        raise ValueError(f"Conflicting labels in original/ORB pair: {pair_key}")

    output_path_by_kind = {
        str(row["image_kind"]): str(row["output_image_path"])
        for row in pair_rows.to_dict(orient="records")
    }
    pair_records.append({
        "pair_key": (str(clip_id), str(source_image_path)),
        "clip_id": str(clip_id),
        "source_image_path": str(source_image_path),
        "class_name": str(class_values[0]),
        "class_code": CLASS_FOLDERS[str(class_values[0])],
        "original_image_path": output_path_by_kind["original"],
        "orb_image_path": output_path_by_kind["orb_transformed"],
    })

# Link video references when any byte-identical model input appears in both.
# This preserves video grouping and prevents exact copies crossing splits.
parent_by_clip = {
    clip_record["clip_id"]: clip_record["clip_id"]
    for clip_record in clip_inventory
}


def find_clip_root(clip_id):
    """Return the canonical root for one content-linked video reference."""
    parent = parent_by_clip[clip_id]
    if parent != clip_id:
        parent_by_clip[clip_id] = find_clip_root(parent)
    return parent_by_clip[clip_id]


def union_clips(left_clip_id, right_clip_id):
    """Merge two video references into one indivisible split unit."""
    left_root = find_clip_root(left_clip_id)
    right_root = find_clip_root(right_clip_id)
    if left_root != right_root:
        parent_by_clip[right_root] = left_root


clip_ids_by_content_hash = defaultdict(list)
class_by_clip_id = {
    clip_record["clip_id"]: clip_record["class_code"]
    for clip_record in clip_inventory
}
for pair_record in pair_records:
    for image_path_key in ("original_image_path", "orb_image_path"):
        image_bytes = Path(pair_record[image_path_key]).read_bytes()
        content_hash = hashlib.sha256(image_bytes).hexdigest()
        clip_ids_by_content_hash[content_hash].append(pair_record["clip_id"])

for content_hash, clip_ids in clip_ids_by_content_hash.items():
    unique_clip_ids = sorted(set(clip_ids))
    if len(unique_clip_ids) < 2:
        continue
    content_classes = {
        class_by_clip_id[clip_id] for clip_id in unique_clip_ids
    }
    if len(content_classes) != 1:
        raise ValueError(
            "Byte-identical content has conflicting class labels: "
            f"{content_hash}, {unique_clip_ids}"
        )
    first_clip_id = unique_clip_ids[0]
    for duplicate_clip_id in unique_clip_ids[1:]:
        union_clips(first_clip_id, duplicate_clip_id)

# Build content-linked video units within each class.
clips_by_root = defaultdict(list)
for clip_record in clip_inventory:
    root_clip_id = find_clip_root(clip_record["clip_id"])
    clips_by_root[root_clip_id].append(clip_record)

split_units_by_class = defaultdict(list)
for root_clip_id, linked_clips in clips_by_root.items():
    class_codes = {clip_record["class_code"] for clip_record in linked_clips}
    if len(class_codes) != 1:
        raise ValueError(f"Content-linked videos have conflicting labels: {root_clip_id}")
    class_code = next(iter(class_codes))
    split_units_by_class[class_code].append(linked_clips)

clip_counts_by_class = {
    class_code: sum(len(unit) for unit in split_units_by_class[class_code])
    for class_code in range(len(CLASS_NAMES))
}
if any(clip_count == 0 for clip_count in clip_counts_by_class.values()):
    raise ValueError(f"At least one class has no videos: {clip_counts_by_class}")


def find_attainable_evaluation_counts(split_units):
    """Return validation/test video-count pairs that preserve complete units."""
    attainable_counts = {(0, 0)}

    for split_unit in split_units:
        unit_size = len(split_unit)
        next_counts = set(attainable_counts)
        for validation_count, test_count in attainable_counts:
            next_counts.add((validation_count + unit_size, test_count))
            next_counts.add((validation_count, test_count + unit_size))
        attainable_counts = next_counts

    return attainable_counts


# Request quotas from the smallest class, then find the nearest quotas that
# every class can achieve without separating byte-identical video references.
smallest_class_clip_count = min(clip_counts_by_class.values())
requested_validation_count = max(
    1,
    round(smallest_class_clip_count * CFG.val_ratio),
)
requested_test_count = max(
    1,
    round(smallest_class_clip_count * CFG.test_ratio),
)

attainable_counts_by_class = []
for class_code in range(len(CLASS_NAMES)):
    class_attainable_counts = find_attainable_evaluation_counts(
        split_units_by_class[class_code]
    )
    attainable_counts_by_class.append(class_attainable_counts)

common_attainable_counts = set.intersection(*attainable_counts_by_class)
valid_common_counts = [
    counts
    for counts in common_attainable_counts
    if counts[0] >= 1
    and counts[1] >= 1
    and all(
        clip_counts_by_class[class_code] - counts[0] - counts[1] >= 1
        for class_code in range(len(CLASS_NAMES))
    )
]
if not valid_common_counts:
    raise ValueError(
        "No class-balanced validation/test quotas preserve content-linked "
        "groups and leave at least one training video per class."
    )


def evaluation_target_score(counts):
    """Rank attainable targets by distance from the requested split ratios."""
    validation_count, test_count = counts
    total_difference = abs(
        validation_count
        + test_count
        - requested_validation_count
        - requested_test_count
    )
    individual_difference = (
        abs(validation_count - requested_validation_count)
        + abs(test_count - requested_test_count)
    )

    # Prefer the larger test set when two targets are equally close.
    return (
        individual_difference,
        total_difference,
        abs(validation_count - requested_validation_count),
        -test_count,
    )


validation_clips_per_class, test_clips_per_class = min(
    valid_common_counts,
    key=evaluation_target_score,
)
print(
    "\nRequested evaluation videos per class: "
    f"{requested_validation_count} validation, {requested_test_count} test"
)
print(
    "Content-safe evaluation videos per class: "
    f"{validation_clips_per_class} validation, "
    f"{test_clips_per_class} test"
)


def allocate_units_to_evaluation_splits(
    split_units,
    validation_video_target,
    test_video_target,
):
    """Allocate whole content-linked units to exact validation and test targets."""
    # Each state stores the unit indices assigned to validation and test.
    allocation_states = {(0, 0): (tuple(), tuple())}

    for unit_index, split_unit in enumerate(split_units):
        unit_size = len(split_unit)
        next_states = dict(allocation_states)

        for counts, selected_indices in allocation_states.items():
            validation_count, test_count = counts
            validation_indices, test_indices = selected_indices

            # Try assigning this complete unit to validation.
            new_validation_count = validation_count + unit_size
            validation_state = (new_validation_count, test_count)
            if (
                new_validation_count <= validation_video_target
                and validation_state not in next_states
            ):
                next_states[validation_state] = (
                    validation_indices + (unit_index,),
                    test_indices,
                )

            # Try assigning this complete unit to test.
            new_test_count = test_count + unit_size
            test_state = (validation_count, new_test_count)
            if new_test_count <= test_video_target and test_state not in next_states:
                next_states[test_state] = (
                    validation_indices,
                    test_indices + (unit_index,),
                )

        allocation_states = next_states

    target_state = (validation_video_target, test_video_target)
    if target_state not in allocation_states:
        unit_sizes = sorted(len(unit) for unit in split_units)
        raise ValueError(
            "Could not jointly allocate exact validation and test video targets "
            "without splitting a byte-identical content group. "
            f"Targets={target_state}, unit sizes={unit_sizes}"
        )

    validation_indices, test_indices = allocation_states[target_state]
    validation_index_set = set(validation_indices)
    test_index_set = set(test_indices)

    validation_units = [
        unit for index, unit in enumerate(split_units)
        if index in validation_index_set
    ]
    test_units = [
        unit for index, unit in enumerate(split_units)
        if index in test_index_set
    ]
    training_units = [
        unit for index, unit in enumerate(split_units)
        if index not in validation_index_set and index not in test_index_set
    ]
    return validation_units, test_units, training_units


# Assign complete content-linked video units without considering their dates.
random_generator = np.random.default_rng(CFG.seed)
split_by_clip = {}
for class_code in range(len(CLASS_NAMES)):
    class_units = list(split_units_by_class[class_code])
    random_generator.shuffle(class_units)

    validation_units, test_units, training_units = (
        allocate_units_to_evaluation_splits(
            class_units,
            validation_clips_per_class,
            test_clips_per_class,
        )
    )

    for split_name, selected_units in (
        ("validation", validation_units),
        ("test", test_units),
        ("train", training_units),
    ):
        for split_unit in selected_units:
            for clip_record in split_unit:
                split_by_clip[clip_record["clip_id"]] = split_name

# Propagate each video's split to all of its original/ORB pairs.
split_by_pair = {}
for pair_record in pair_records:
    pair_record["split"] = split_by_clip[pair_record["clip_id"]]
    split_by_pair[pair_record["pair_key"]] = pair_record["split"]

train_clip_set = {clip_id for clip_id, split in split_by_clip.items() if split == "train"}
val_clip_set = {clip_id for clip_id, split in split_by_clip.items() if split == "validation"}
test_clip_set = {clip_id for clip_id, split in split_by_clip.items() if split == "test"}
assert train_clip_set.isdisjoint(val_clip_set)
assert train_clip_set.isdisjoint(test_clip_set)
assert val_clip_set.isdisjoint(test_clip_set)

train_pair_set = {
    pair_record["pair_key"] for pair_record in pair_records
    if pair_record["split"] == "train"
}
val_pair_set = {
    pair_record["pair_key"] for pair_record in pair_records
    if pair_record["split"] == "validation"
}
test_pair_set = {
    pair_record["pair_key"] for pair_record in pair_records
    if pair_record["split"] == "test"
}
assert train_pair_set.isdisjoint(val_pair_set)
assert train_pair_set.isdisjoint(test_pair_set)
assert val_pair_set.isdisjoint(test_pair_set)

print("\nVideos per split:")
print(f"  Train:      {len(train_clip_set):>5,}")
print(f"  Validation: {len(val_clip_set):>5,}")
print(f"  Test:       {len(test_clip_set):>5,}")
print("\nOriginal/ORB pairs per split:")
print(f"  Train:      {len(train_pair_set):>5,}")
print(f"  Validation: {len(val_pair_set):>5,}")
print(f"  Test:       {len(test_pair_set):>5,}")


---

## Chapter 3: Prepare frame samples and temporal rules

No videos are decoded and no frames are selected for training here. The notebook reads the checked `stoppage_detection_and_classification/cnn_classifier/data/training_frames` dataset and assigns each complete video reference to a class-balanced split, ignoring dates.

The temporal functions in this section are used only when classifying new videos after training.


In [ ]:
INFERENCE_AGGREGATION_VERSION = "class-specific-temporal-v3-no-bottle"


def apply_video_orientation(frames, rotate_180):
    """Apply the known historical camera correction for live inference."""
    if not rotate_180:
        return frames
    return [np.ascontiguousarray(np.rot90(frame, 2)) for frame in frames]


def select_inference_frame_indices(frame_count, source_fps, sample_fps):
    """Select every frame or a consistent FPS sample without using content."""
    if frame_count <= 0:
        raise ValueError("Cannot select frames from an empty clip")
    if source_fps <= 0:
        raise ValueError("Source FPS must be positive")
    if sample_fps is None or sample_fps >= source_fps:
        return list(range(frame_count))
    if sample_fps <= 0:
        raise ValueError("sample_fps must be None or positive")

    frame_step = max(1, int(round(source_fps / sample_fps)))
    return list(range(0, frame_count, frame_step))



def _maximum_consecutive_frames(boolean_values):
    """Return the longest consecutive True run in one temporal window."""

    longest_run = 0
    current_run = 0
    for is_active in boolean_values:
        if is_active:
            current_run += 1
            longest_run = max(longest_run, current_run)
        else:
            current_run = 0
    return longest_run


def _frames_for_seconds(seconds, effective_sample_fps, minimum_frames=1):
    """Convert a duration to frames without floating-point boundary inflation."""

    frame_count = int(np.ceil(seconds * effective_sample_fps - 1e-9))
    return max(minimum_frames, frame_count)


def calculate_temporal_clip_decision(
    frame_probabilities: np.ndarray,
    frame_indices: list[int],
    source_fps: float,
    class_names: list[str],
    rolling_window_seconds: float,
    fallen_threshold: float,
    falling_threshold: float,
    normal_threshold: float = 0.75,
    fault_activity_threshold: float = 0.50,
    falling_peak_threshold: float = 0.80,
    persistent_seconds: float = 0.3,
    min_falling_seconds: float = 0.10,
    peak_support_seconds: float = 0.10,
    normal_clear_seconds: float = 0.5,
    fallen_clear_seconds: float = 0.75,
) -> dict:
    """Apply class-specific FPS-aware temporal rules to one classified clip."""

    probabilities = np.asarray(frame_probabilities, dtype=np.float64)
    indices = np.asarray(frame_indices, dtype=np.int64)
    if len(probabilities) == 0 or len(probabilities) != len(indices):
        raise ValueError("Frame probabilities and indices must be non-empty and aligned")
    if source_fps <= 0:
        raise ValueError("source_fps must be positive")

    no_bottle_index = class_names.index("no_bottle")
    normal_index = class_names.index("normal")
    fallen_index = class_names.index("fallen_before_entry")
    falling_index = class_names.index("fallen_in_view")
    frame_times = indices / float(source_fps)

    # Use the actual classified-frame rate, including any inference sampling.
    if len(indices) == 1:
        effective_sample_fps = float(source_fps)
    else:
        positive_index_steps = np.diff(indices)
        positive_index_steps = positive_index_steps[positive_index_steps > 0]
        effective_sample_fps = (
            float(source_fps / np.median(positive_index_steps))
            if len(positive_index_steps)
            else float(source_fps)
        )

    window_size = _frames_for_seconds(rolling_window_seconds, effective_sample_fps)
    persistent_size = _frames_for_seconds(persistent_seconds, effective_sample_fps)
    min_falling_frames = _frames_for_seconds(min_falling_seconds, effective_sample_fps, minimum_frames=2)
    support_radius_frames = _frames_for_seconds(peak_support_seconds, effective_sample_fps)
    normal_clear_frames = _frames_for_seconds(normal_clear_seconds, effective_sample_fps, minimum_frames=2)
    fallen_clear_frames = _frames_for_seconds(fallen_clear_seconds, effective_sample_fps, minimum_frames=2)
    window_size = min(window_size, len(probabilities))
    persistent_size = min(persistent_size, window_size)

    decision_windows = []
    current_state = "normal"
    stable_normal_frames = 0
    observed_falling_in_view = False

    for window_end in range(window_size - 1, len(probabilities)):
        window_start = window_end - window_size + 1
        history = probabilities[window_start : window_end + 1]
        persistent_history = history[-persistent_size:]

        no_bottle_values = history[:, no_bottle_index]
        normal_values = history[:, normal_index]
        fallen_values = persistent_history[:, fallen_index]
        falling_values = history[:, falling_index]

        no_bottle_mean = float(np.mean(no_bottle_values))
        normal_mean = float(np.mean(normal_values))
        safe_classification = (
            "no_bottle" if no_bottle_mean >= normal_mean else "normal"
        )
        safe_mean = max(no_bottle_mean, normal_mean)
        safe_values = (
            no_bottle_values if safe_classification == "no_bottle" else normal_values
        )
        fallen_median = float(np.median(fallen_values))
        falling_peak_relative = int(np.argmax(falling_values))
        falling_peak = float(falling_values[falling_peak_relative])
        falling_peak_position = window_start + falling_peak_relative

        falling_active = falling_values >= falling_threshold
        falling_consecutive_frames = _maximum_consecutive_frames(falling_active)

        local_start = max(0, falling_peak_relative - support_radius_frames)
        local_stop = min(len(falling_values), falling_peak_relative + support_radius_frames + 1)
        falling_support_frames = int(np.count_nonzero(
            falling_values[local_start:local_stop] >= fault_activity_threshold
        ))
        falling_frames_above_activity = int(np.count_nonzero(
            falling_values >= fault_activity_threshold
        ))
        fallen_frames_above_activity = int(np.count_nonzero(
            history[:, fallen_index] >= fault_activity_threshold
        ))

        # Fault evidence takes precedence over normal evidence.
        if fallen_median >= fallen_threshold:
            candidate_classification = "fallen_before_entry"
            candidate_score = fallen_median
            persistent_start = window_end - persistent_size + 1
            selected_relative = int(np.argmax(
                probabilities[persistent_start : window_end + 1, fallen_index]
            ))
            selected_frame_position = persistent_start + selected_relative
        elif falling_consecutive_frames >= min_falling_frames:
            candidate_classification = "fallen_in_view"
            candidate_score = falling_peak
            selected_frame_position = falling_peak_position
        elif (
            falling_peak >= falling_peak_threshold
            and falling_support_frames >= 2
        ):
            candidate_classification = "fallen_in_view"
            candidate_score = falling_peak
            selected_frame_position = falling_peak_position
        elif (
            safe_mean >= normal_threshold
            and falling_frames_above_activity < 2
            and fallen_frames_above_activity < 2
        ):
            candidate_classification = safe_classification
            candidate_score = safe_mean
            selected_frame_position = window_start + int(np.argmax(safe_values))
        else:
            candidate_classification = "uncertain"
            candidate_score = float(np.max(history[-1]))
            selected_frame_position = window_end

        # Preserve how the fault began, and require stable normal evidence to clear it.
        if candidate_classification == "fallen_in_view":
            observed_falling_in_view = True
            current_state = "fallen_in_view"
            stable_normal_frames = 0
        elif candidate_classification == "fallen_before_entry":
            if not observed_falling_in_view:
                current_state = "fallen_before_entry"
            stable_normal_frames = 0
        elif candidate_classification in {"normal", "no_bottle"}:
            stable_normal_frames += 1
            required_clear_frames = (
                fallen_clear_frames
                if current_state == "fallen_before_entry"
                else normal_clear_frames
            )
            if (
                current_state in {"normal", "no_bottle", "uncertain"}
                or stable_normal_frames >= required_clear_frames
            ):
                current_state = candidate_classification
        else:
            stable_normal_frames = 0
            if current_state in {"normal", "no_bottle"}:
                current_state = "uncertain"

        decision_windows.append({
            "candidate_classification": candidate_classification,
            "state": current_state,
            "score": candidate_score,
            "window_start_position": window_start,
            "window_end_position": window_end,
            "selected_frame_position": selected_frame_position,
            "no_bottle_mean": no_bottle_mean,
            "normal_mean": normal_mean,
            "fallen_median": fallen_median,
            "falling_peak": falling_peak,
            "falling_consecutive_frames": falling_consecutive_frames,
            "falling_support_frames": falling_support_frames,
        })

    falling_windows = [
        item for item in decision_windows
        if item["candidate_classification"] == "fallen_in_view"
    ]
    fallen_windows = [
        item for item in decision_windows
        if item["candidate_classification"] == "fallen_before_entry"
    ]
    safe_windows = [
        item for item in decision_windows
        if item["candidate_classification"] in {"normal", "no_bottle"}
    ]

    # A detected in-view fall remains the clip's event origin even after it settles.
    if falling_windows:
        prediction = "fallen_in_view"
        selected_window = max(falling_windows, key=lambda item: item["score"])
    elif fallen_windows:
        prediction = "fallen_before_entry"
        selected_window = max(fallen_windows, key=lambda item: item["score"])
    elif safe_windows:
        selected_window = max(safe_windows, key=lambda item: item["score"])
        prediction = selected_window["candidate_classification"]
    else:
        prediction = "uncertain"
        selected_window = max(decision_windows, key=lambda item: item["score"])

    peak_frame_position = int(selected_window["selected_frame_position"])
    peak_window_position = int(selected_window["window_start_position"])
    peak_window_end_position = int(selected_window["window_end_position"])
    confidence = float(selected_window["score"])

    strongest_fallen_window = max(decision_windows, key=lambda item: item["fallen_median"])
    strongest_falling_window = max(decision_windows, key=lambda item: item["falling_peak"])
    temporal_scores = {
        "no_bottle": max(item["no_bottle_mean"] for item in decision_windows),
        "normal": max(item["normal_mean"] for item in decision_windows),
        "fallen_before_entry": strongest_fallen_window["fallen_median"],
        "fallen_in_view": strongest_falling_window["falling_peak"],
    }

    detection_events = [
        item for item in decision_windows
        if item["candidate_classification"] in {"fallen_before_entry", "fallen_in_view"}
    ]

    return {
        "prediction": prediction,
        "confidence": confidence,
        "temporal_scores": temporal_scores,
        "peak_anomaly_score": confidence,
        "peak_anomaly_class": prediction,
        "peak_frame_anomaly_probability": float(
            probabilities[peak_frame_position].max()
        ),
        "peak_anomaly_frame_index": int(indices[peak_frame_position]),
        "peak_anomaly_frame_time_seconds": float(frame_times[peak_frame_position]),
        "effective_sample_fps": effective_sample_fps,
        "rolling_window_frame_count": int(window_size),
        "persistent_frame_count": int(persistent_size),
        "min_falling_frame_count": int(min_falling_frames),
        "peak_window_start_frame": int(indices[peak_window_position]),
        "peak_window_end_frame": int(indices[peak_window_end_position]),
        "peak_window_start_seconds": float(frame_times[peak_window_position]),
        "peak_window_end_seconds": float(frame_times[peak_window_end_position]),
        "strongest_entry_window_start_frame": int(indices[strongest_fallen_window["window_start_position"]]),
        "strongest_entry_window_end_frame": int(indices[strongest_fallen_window["window_end_position"]]),
        "strongest_entry_window_start_seconds": float(frame_times[strongest_fallen_window["window_start_position"]]),
        "strongest_entry_window_end_seconds": float(frame_times[strongest_fallen_window["window_end_position"]]),
        "strongest_fall_window_start_frame": int(indices[strongest_falling_window["window_start_position"]]),
        "strongest_fall_window_end_frame": int(indices[strongest_falling_window["window_end_position"]]),
        "strongest_fall_window_start_seconds": float(frame_times[strongest_falling_window["window_start_position"]]),
        "strongest_fall_window_end_seconds": float(frame_times[strongest_falling_window["window_end_position"]]),
        "decision_windows": decision_windows,
        "detection_events": detection_events,
        "preview_position": peak_frame_position,
        "strongest_entry_window": {
            "start_frame": int(indices[strongest_fallen_window["window_start_position"]]),
            "end_frame": int(indices[strongest_fallen_window["window_end_position"]]),
            "start_seconds": float(frame_times[strongest_fallen_window["window_start_position"]]),
            "end_seconds": float(frame_times[strongest_fallen_window["window_end_position"]]),
        },
        "strongest_fall_window": {
            "start_frame": int(indices[strongest_falling_window["window_start_position"]]),
            "end_frame": int(indices[strongest_falling_window["window_end_position"]]),
            "start_seconds": float(frame_times[strongest_falling_window["window_start_position"]]),
            "end_seconds": float(frame_times[strongest_falling_window["window_end_position"]]),
        },
    }


# Attach every prepared row to the split assigned to its original/ORB pair.
record_by_clip_id = {record["clip_id"]: record for record in clip_inventory}
all_samples = []
for manifest_row in frame_manifest_df.to_dict(orient="records"):
    clip_id = str(manifest_row["clip_id"])
    source_image_path = str(manifest_row["source_image_path"])
    pair_key = (clip_id, source_image_path)
    if clip_id not in record_by_clip_id:
        raise ValueError(f"Manifest row has no clip record: {clip_id}")
    if pair_key not in split_by_pair:
        raise ValueError(f"Manifest row has no pair split: {pair_key}")

    clip_record = record_by_clip_id[clip_id]
    all_samples.append({
        "frame_path": Path(manifest_row["output_image_path"]),
        "source_frame_path": Path(source_image_path),
        "pair_key": pair_key,
        "label": clip_record["class_code"],
        "class_name": clip_record["class_name"],
        "clip_id": clip_id,
        "clip_name": clip_record["video_path"].name,
        "split": split_by_pair[pair_key],
        "image_kind": str(manifest_row["image_kind"]),
        "is_orb_transformed": str(manifest_row["image_kind"]) == "orb_transformed",
    })

if not all_samples:
    raise ValueError("The 50/50 frame manifest produced no samples")

# Prove that every split is exactly 50% original and 50% ORB-transformed.
kind_counts_by_split = Counter(
    (sample["split"], sample["image_kind"])
    for sample in all_samples
)
for split_name in ("train", "validation", "test"):
    original_count = kind_counts_by_split[(split_name, "original")]
    orb_count = kind_counts_by_split[(split_name, "orb_transformed")]
    if original_count == 0 or original_count != orb_count:
        raise ValueError(
            f"{split_name} is not exactly 50/50 original/ORB: "
            f"{original_count} original, {orb_count} ORB"
        )

# Prove that each original/ORB pair remains entirely within one split.
splits_by_source_frame = defaultdict(set)
splits_by_clip = defaultdict(set)
for sample in all_samples:
    splits_by_source_frame[sample["pair_key"]].add(sample["split"])
    splits_by_clip[sample["clip_id"]].add(sample["split"])
if any(len(splits) != 1 for splits in splits_by_source_frame.values()):
    raise ValueError("At least one original/ORB pair crosses dataset splits")

# Prove that all frames from one video reference remain in one split.
clips_crossing_splits = sum(len(splits) > 1 for splits in splits_by_clip.values())
if clips_crossing_splits:
    raise ValueError(f"{clips_crossing_splits} videos cross dataset splits")

# Prove that one source path cannot be assigned to multiple clips.
clip_ids_by_source_path = defaultdict(set)
for sample in all_samples:
    source_path = str(sample["source_frame_path"].resolve())
    clip_ids_by_source_path[source_path].add(sample["clip_id"])
shared_source_paths = {
    source_path: clip_ids
    for source_path, clip_ids in clip_ids_by_source_path.items()
    if len(clip_ids) > 1
}
if shared_source_paths:
    first_shared_path = next(iter(shared_source_paths.items()))
    raise ValueError(f"A source image belongs to multiple clips: {first_shared_path}")

# Detect byte-identical model inputs assigned to different splits.
records_by_content_hash = defaultdict(list)
for sample in all_samples:
    frame_bytes = sample["frame_path"].read_bytes()
    content_hash = hashlib.sha256(frame_bytes).hexdigest()
    records_by_content_hash[content_hash].append({
        "split": sample["split"],
        "clip_id": sample["clip_id"],
        "frame_path": str(sample["frame_path"]),
    })

cross_split_content_hashes = {
    content_hash: records
    for content_hash, records in records_by_content_hash.items()
    if len({record["split"] for record in records}) > 1
}
if cross_split_content_hashes:
    first_duplicate = next(iter(cross_split_content_hashes.items()))
    raise ValueError(
        "Byte-identical frame content crosses dataset splits: "
        f"{first_duplicate}"
    )

leakage_audit_summary = {
    "split_policy": "class-balanced video-reference split; exact-content-linked videos kept together; dates ignored",
    "video_reference_grouping": True,
    "clips_crossing_splits": clips_crossing_splits,
    "original_orb_pairs_crossing_splits": 0,
    "source_paths_shared_between_clips": 0,
    "exact_image_hashes_crossing_splits": 0,
}
print("Pair-level split audit:")
for check_name, failure_count in leakage_audit_summary.items():
    print(f"  {check_name}: {failure_count}")

frame_counts = Counter(sample["label"] for sample in all_samples)
print(f"\nPrepared frame samples: {len(all_samples):,}")
for class_code, class_name in enumerate(CLASS_NAMES):
    print(f"  {class_name:<20s} {frame_counts[class_code]:>6,} frames")
print("\nExact original/ORB counts by split:")
for split_name in ("train", "validation", "test"):
    original_count = kind_counts_by_split[(split_name, "original")]
    orb_count = kind_counts_by_split[(split_name, "orb_transformed")]
    print(f"  {split_name:<10s} {original_count:>5,} original + {orb_count:>5,} ORB")

# Require exactly equal video counts per class in validation and test.
clip_counts_by_split_and_class = Counter(
    (split_by_clip[clip_record["clip_id"]], clip_record["class_code"])
    for clip_record in clip_inventory
)
for split_name in ("validation", "test"):
    evaluation_class_counts = [
        clip_counts_by_split_and_class[(split_name, class_code)]
        for class_code in range(len(CLASS_NAMES))
    ]
    if len(set(evaluation_class_counts)) != 1:
        raise ValueError(
            f"{split_name} video counts are not class-balanced: "
            f"{evaluation_class_counts}"
        )


In [ ]:
# Dataset overview: manifest-frame class distribution and examples.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

class_frame_totals = [frame_counts[index] for index in range(len(CLASS_NAMES))]
sns.barplot(x=CLASS_NAMES, y=class_frame_totals, ax=axes[0], color="steelblue")
axes[0].set_title("50/50 CNN frames per class")
axes[0].set_ylabel("Frames")
axes[0].tick_params(axis="x", rotation=20)

# Show one deterministic original frame from each class.
axes[1].axis("off")
example_fig, example_axes = plt.subplots(1, len(CLASS_NAMES), figsize=(15, 5))
example_fig.suptitle("Example checked original frames")
for class_code, axis in enumerate(np.atleast_1d(example_axes)):
    example_sample = next(
        sample
        for sample in all_samples
        if sample["label"] == class_code and sample["image_kind"] == "original"
    )
    axis.imshow(PILImage.open(example_sample["frame_path"]).convert("RGB"))
    axis.set_title(CLASS_NAMES[class_code])
    axis.axis("off")

plt.tight_layout()
plt.show()


---

## Chapter 4: Aspect-preserving image preprocessing

Training, validation, testing and S3 inference use the same aspect-preserving letterbox policy. The image is resized to fit inside `224?224`, then padded with the ImageNet mean colour instead of being stretched. Training alone adds fixed-camera-safe geometric and colour augmentation.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
MODEL_HANDOFF_SCHEMA_VERSION = 3
IMAGE_PREPROCESSING_VERSION = "aspect-preserving-letterbox-v1"
LETTERBOX_FILL = tuple(round(channel * 255) for channel in IMAGENET_MEAN)


def letterbox_image(image, output_size=CFG.img_size):
    """Resize an image without distortion and pad it to a square."""
    image = image.convert("RGB")
    resized = ImageOps.contain(
        image,
        (output_size, output_size),
        method=PILImage.Resampling.BILINEAR,
    )
    canvas = PILImage.new("RGB", (output_size, output_size), LETTERBOX_FILL)
    left = (output_size - resized.width) // 2
    top = (output_size - resized.height) // 2
    canvas.paste(resized, (left, top))
    return canvas


# Fixed camera perspective means flips and random crops are intentionally omitted.
train_transforms = T.Compose([
    letterbox_image,
    T.RandomRotation(degrees=10, fill=LETTERBOX_FILL),
    T.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),
        scale=(0.92, 1.08),
        fill=LETTERBOX_FILL,
    ),
    T.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.2,
        hue=0.03,
    ),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.1)),
])

val_test_transforms = T.Compose([
    letterbox_image,
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f"Image preprocessing: {IMAGE_PREPROCESSING_VERSION}")
print(f"Letterbox fill RGB:  {LETTERBOX_FILL}")
print(f"Train transforms:    {len(train_transforms.transforms)} operations")
print(f"Val/test transforms: {len(val_test_transforms.transforms)} operations")


# Report whether the existing checkpoint was produced by this exact data contract.
existing_metadata_path = CFG.output_dir / "model_metadata.json"
existing_checkpoint_is_current = False
if existing_metadata_path.is_file() and CFG.model_save_path.is_file():
    with existing_metadata_path.open("r", encoding="utf-8") as metadata_file:
        existing_metadata = json.load(metadata_file)

    current_manifest_hash = hashlib.sha256(
        CFG.frame_manifest_path.read_bytes()
    ).hexdigest()
    existing_checkpoint_is_current = (
        existing_metadata.get("handoff_schema_version")
        == MODEL_HANDOFF_SCHEMA_VERSION
        and existing_metadata.get("image_preprocessing_version")
        == IMAGE_PREPROCESSING_VERSION
        and existing_metadata.get("training_manifest_sha256")
        == current_manifest_hash
        and Path(existing_metadata.get("frame_dataset_root", "")).resolve()
        == CFG.frame_dataset_root.resolve()
    )

if existing_checkpoint_is_current:
    print("Existing checkpoint matches the current manifest and preprocessing")
else:
    print(
        "Existing checkpoint is stale or uses stretched-image preprocessing; "
        "run both training phases and the final metadata cell before inference."
    )


In [ ]:
# Preview training augmentation on one manifest frame.
preview_sample = all_samples[0]
preview_image = PILImage.open(preview_sample["frame_path"]).convert("RGB")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f"Training augmentation preview — {preview_sample['class_name']}")
for axis in axes.flat:
    augmented = train_transforms(preview_image)
    augmented = augmented * torch.tensor(IMAGENET_STD)[:, None, None]
    augmented = augmented + torch.tensor(IMAGENET_MEAN)[:, None, None]
    axis.imshow(augmented.clamp(0, 1).permute(1, 2, 0))
    axis.axis("off")
plt.tight_layout()
plt.show()

---

## Chapter 5: Build balanced data loaders

Validation and test reserve the same number of video references from every class. Dates are ignored, but each video and any video linked by byte-identical content remain in one split.

In [ ]:
class FrameClassificationDataset(Dataset):
    """Load prepared manifest frames and their clip labels."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = PILImage.open(sample["frame_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, sample["label"]


# Every video, source frame, original, and ORB copy remains in one split.
assert train_pair_set.isdisjoint(val_pair_set)
assert train_pair_set.isdisjoint(test_pair_set)
assert val_pair_set.isdisjoint(test_pair_set)

train_samples = [sample for sample in all_samples if sample["split"] == "train"]
val_samples = [sample for sample in all_samples if sample["split"] == "validation"]
test_samples = [sample for sample in all_samples if sample["split"] == "test"]

train_dataset = FrameClassificationDataset(train_samples, transform=train_transforms)
val_dataset = FrameClassificationDataset(val_samples, transform=val_test_transforms)
test_dataset = FrameClassificationDataset(test_samples, transform=val_test_transforms)

# Balance training batches with one mechanism: weighted sampling.
train_labels = np.array([sample["label"] for sample in train_samples])
train_class_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
if np.any(train_class_counts == 0):
    raise ValueError(f"A training class has no samples: {train_class_counts}")

weight_per_class = 1.0 / train_class_counts.astype(float)
sample_weights = weight_per_class[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).double(),
    num_samples=len(train_dataset),
    replacement=True,
)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    sampler=sampler,
    num_workers=CFG.num_workers,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=pin_memory,
)

print("Videos per split:")
print(f"  Train:      {len(train_clip_set):>5,}")
print(f"  Validation: {len(val_clip_set):>5,}")
print(f"  Test:       {len(test_clip_set):>5,}")
print("\nOriginal/ORB pairs per split:")
print(f"  Train:      {len(train_pair_set):>5,}")
print(f"  Validation: {len(val_pair_set):>5,}")
print(f"  Test:       {len(test_pair_set):>5,}")
print("\nManifest frames per split:")
print(f"  Train:      {len(train_dataset):>6,}")
print(f"  Validation: {len(val_dataset):>6,}")
print(f"  Test:       {len(test_dataset):>6,}")

print("\nClass distribution per split:")
for split_name, split_samples in [
    ("Train", train_samples),
    ("Val", val_samples),
    ("Test", test_samples),
]:
    counts = Counter(sample["label"] for sample in split_samples)
    details = " | ".join(
        f"{class_name}: {counts[class_code]:,}"
        for class_code, class_name in enumerate(CLASS_NAMES)
    )
    print(f"  {split_name:<5s} {details}")

In [ ]:
# The sampler already balances classes, so the loss must remain unweighted.
print("Balancing strategy:")
print("  1. WeightedRandomSampler balances training batches")
print("  2. CrossEntropyLoss is unweighted to avoid double correction")
print("  3. Macro F1 and balanced accuracy are reported at clip level")

---

## Chapter 6: Build the four-class ResNet50

This is the same architecture and progressive-unfreezing strategy as the reference notebook.

In [ ]:
def build_model(num_classes=len(CLASS_NAMES), dropout=0.5):
    """Build an ImageNet-pretrained ResNet50 with a custom classifier head."""
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    # Phase 1 trains only the new classification head.
    for parameter in model.parameters():
        parameter.requires_grad = False

    input_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(input_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    return model


def unfreeze_layers(model, layer_names):
    """Unfreeze named ResNet stages and return their parameter count."""
    unfrozen_parameter_count = 0
    for parameter_name, parameter in model.named_parameters():
        if any(parameter_name.startswith(layer_name) for layer_name in layer_names):
            parameter.requires_grad = True
            unfrozen_parameter_count += parameter.numel()
    return unfrozen_parameter_count


def count_parameters(model):
    """Return trainable and total parameter counts."""
    total_count = sum(parameter.numel() for parameter in model.parameters())
    trainable_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return trainable_count, total_count


model = build_model().to(DEVICE)
trainable_count, total_count = count_parameters(model)

print(f"Model: ResNet50 + custom head ({len(CLASS_NAMES)} classes)")
print(f"Classes:          {CLASS_NAMES}")
print(f"Total params:     {total_count:>12,}")
print(f"Trainable params: {trainable_count:>12,} ({trainable_count / total_count:.1%})")

---

## Chapter 7: Train and select the best checkpoint

In [ ]:
@dataclass
class TrainHistory:
    """Store per-epoch metrics for plotting."""

    train_loss: list[float]
    val_loss: list[float]
    train_accuracy: list[float]
    val_accuracy: list[float]
    val_clip_f1_macro: list[float]
    phase_boundary: int = 0


def autocast_context(device):
    """Enable mixed precision only when CUDA is active."""
    return torch.amp.autocast(
        device_type=device.type,
        enabled=device.type == "cuda",
    )


def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    """Train for one epoch and return average loss and frame accuracy."""
    model.train()
    running_loss = 0.0
    correct_count = 0
    sample_count = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context(device):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct_count += logits.argmax(dim=1).eq(labels).sum().item()
        sample_count += batch_size

    return running_loss / sample_count, correct_count / sample_count


@torch.no_grad()
def evaluate_validation(model, loader, criterion, device):
    """Return frame metrics and class-balanced clip macro F1."""
    model.eval()
    running_loss = 0.0
    correct_count = 0
    sample_count = 0
    probability_batches = []
    label_batches = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast_context(device):
            logits = model(images)
            loss = criterion(logits, labels)

        probabilities = torch.softmax(logits, dim=1)
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct_count += logits.argmax(dim=1).eq(labels).sum().item()
        sample_count += batch_size
        probability_batches.append(probabilities.float().cpu().numpy())
        label_batches.append(labels.cpu().numpy())

    frame_probabilities = np.concatenate(probability_batches, axis=0)
    frame_labels = np.concatenate(label_batches, axis=0)

    # Average each validation video's frames before calculating macro F1.
    frame_positions_by_clip = defaultdict(list)
    for frame_position, sample in enumerate(loader.dataset.samples):
        frame_positions_by_clip[sample["clip_id"]].append(frame_position)

    clip_labels = []
    clip_predictions = []
    for frame_positions in frame_positions_by_clip.values():
        labels_for_clip = set(frame_labels[frame_positions].tolist())
        if len(labels_for_clip) != 1:
            raise ValueError("A validation clip contains conflicting labels.")

        mean_probabilities = frame_probabilities[frame_positions].mean(axis=0)
        clip_labels.append(next(iter(labels_for_clip)))
        clip_predictions.append(int(np.argmax(mean_probabilities)))

    clip_f1_macro = f1_score(
        clip_labels,
        clip_predictions,
        labels=np.arange(len(CLASS_NAMES)),
        average="macro",
        zero_division=0,
    )
    return (
        running_loss / sample_count,
        correct_count / sample_count,
        float(clip_f1_macro),
    )


def run_training(model, train_loader, val_loader, criterion, device, cfg):
    """Train the head, fine-tune deeper layers, and reload the best checkpoint."""
    history = TrainHistory([], [], [], [], [])
    best_validation_clip_f1 = float("-inf")
    best_validation_loss = float("inf")
    scaler = torch.amp.GradScaler(
        device.type,
        enabled=device.type == "cuda",
    )
    total_epochs = cfg.epochs_head + cfg.epochs_finetune

    # Each phase gets its own optimizer and cosine schedule.
    phases = [
        {
            "name": "classifier head",
            "epochs": cfg.epochs_head,
            "learning_rate": cfg.lr_head,
            "unfreeze": [],
        },
        {
            "name": "layer3 + layer4 fine-tuning",
            "epochs": cfg.epochs_finetune,
            "learning_rate": cfg.lr_finetune,
            "unfreeze": ["layer3", "layer4"],
        },
    ]

    completed_epochs = 0
    for phase_index, phase in enumerate(phases, start=1):
        if phase["unfreeze"]:
            unfrozen_count = unfreeze_layers(model, phase["unfreeze"])
            history.phase_boundary = completed_epochs
            print(f"Unfrozen parameters: {unfrozen_count:,}")

        print("=" * 72)
        print(f"PHASE {phase_index}: {phase['name']}")
        print(
            f"Epochs: {phase['epochs']} | "
            f"Learning rate: {phase['learning_rate']:.2e}"
        )
        print("=" * 72)

        optimizer = optim.AdamW(
            (parameter for parameter in model.parameters() if parameter.requires_grad),
            lr=phase["learning_rate"],
            weight_decay=cfg.weight_decay,
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=phase["epochs"],
            eta_min=phase["learning_rate"] * 0.01,
        )
        patience_counter = 0

        for phase_epoch in range(1, phase["epochs"] + 1):
            train_loss, train_accuracy = train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device,
                scaler,
            )
            (
                validation_loss,
                validation_accuracy,
                validation_clip_f1,
            ) = evaluate_validation(
                model,
                val_loader,
                criterion,
                device,
            )
            scheduler.step()
            completed_epochs += 1

            history.train_loss.append(train_loss)
            history.val_loss.append(validation_loss)
            history.train_accuracy.append(train_accuracy)
            history.val_accuracy.append(validation_accuracy)
            history.val_clip_f1_macro.append(validation_clip_f1)

            checkpoint_marker = ""
            f1_improved = validation_clip_f1 > best_validation_clip_f1 + 1e-9
            f1_tied = abs(validation_clip_f1 - best_validation_clip_f1) <= 1e-9
            loss_breaks_tie = f1_tied and validation_loss < best_validation_loss
            if f1_improved or loss_breaks_tie:
                best_validation_clip_f1 = validation_clip_f1
                best_validation_loss = validation_loss
                torch.save(model.state_dict(), cfg.model_save_path)
                patience_counter = 0
                checkpoint_marker = " saved"
            else:
                patience_counter += 1

            current_learning_rate = scheduler.get_last_lr()[0]
            print(
                f"[{completed_epochs:>2}/{total_epochs}] "
                f"train_loss={train_loss:.4f} "
                f"val_loss={validation_loss:.4f} "
                f"train_acc={train_accuracy:.3f} "
                f"val_acc={validation_accuracy:.3f} "
                f"val_clip_f1={validation_clip_f1:.3f} "
                f"lr={current_learning_rate:.2e}{checkpoint_marker}"
            )

            # Match the reference notebook: early stopping applies during fine-tuning.
            if phase_index == 2 and patience_counter >= cfg.patience:
                print(f"Early stopping after {cfg.patience} unimproved epochs")
                break

    model.load_state_dict(
        torch.load(cfg.model_save_path, map_location=device, weights_only=True)
    )
    print(
        f"\nBest validation clip macro F1: "
        f"{best_validation_clip_f1:.4f}"
    )
    print(f"Best validation loss tie-break: {best_validation_loss:.4f}")
    print(f"Best model loaded from: {cfg.model_save_path}")
    return model, history


print("Training functions defined")

In [ ]:
# Use one balancing mechanism: balanced sampling with an unweighted loss.
criterion = nn.CrossEntropyLoss()

model, history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    cfg=CFG,
)

In [ ]:
def plot_training_curves(history, save_path=None):
    """Plot loss, frame accuracy, and validation clip macro F1."""
    epochs = np.arange(1, len(history.train_loss) + 1)
    best_epoch = int(np.argmax(history.val_clip_f1_macro)) + 1

    figure, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize=(16, 6))
    figure.suptitle("Training and validation curves", fontsize=16, fontweight="bold")

    loss_axis.plot(epochs, history.train_loss, "o-", label="Train loss", markersize=4)
    loss_axis.plot(epochs, history.val_loss, "s-", label="Validation loss", markersize=4)
    loss_axis.axvline(best_epoch, color="green", linestyle=":", label=f"Best epoch {best_epoch}")
    loss_axis.set_xlabel("Epoch")
    loss_axis.set_ylabel("Loss")
    loss_axis.grid(alpha=0.3)
    loss_axis.legend()

    accuracy_axis.plot(epochs, history.train_accuracy, "o-", label="Train accuracy", markersize=4)
    accuracy_axis.plot(epochs, history.val_accuracy, "s-", label="Validation frame accuracy", markersize=4)
    accuracy_axis.plot(
        epochs,
        history.val_clip_f1_macro,
        "^-",
        label="Validation clip macro F1",
        markersize=4,
    )
    accuracy_axis.set_xlabel("Epoch")
    accuracy_axis.set_ylabel("Frame accuracy")
    accuracy_axis.set_ylim(0, 1.05)
    accuracy_axis.grid(alpha=0.3)
    accuracy_axis.legend()

    if history.phase_boundary > 0:
        for axis in (loss_axis, accuracy_axis):
            axis.axvline(
                history.phase_boundary + 0.5,
                color="gray",
                linestyle="--",
                alpha=0.7,
                label="Fine-tuning starts",
            )

    plt.tight_layout()
    if save_path is not None:
        figure.savefig(save_path, bbox_inches="tight")
    return figure


training_curve_path = CFG.output_dir / "training_curves.png"
plot_training_curves(history, training_curve_path)
plt.show()
print(f"Saved: {training_curve_path}")

---

## Chapter 8: Evaluate held-out frames and clips

The CNN predicts every prepared manifest frame in the held-out split. The final clip prediction is the class with the highest mean probability across that clip's 50/50 original and ORB frames.


In [ ]:
@torch.no_grad()
def get_frame_predictions(model, loader, device):
    """Return labels, predictions, and probabilities in dataset order."""
    model.eval()
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for images, labels in tqdm(loader, desc="Test frames", leave=False):
        images = images.to(device, non_blocking=True)
        with autocast_context(device):
            logits = model(images)

        probabilities = torch.softmax(logits, dim=1)
        all_labels.append(labels.numpy())
        all_predictions.append(logits.argmax(dim=1).cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())

    return (
        np.concatenate(all_labels),
        np.concatenate(all_predictions),
        np.concatenate(all_probabilities),
    )


def aggregate_clip_predictions(samples, frame_labels, frame_probabilities):
    """Average frame probabilities into one prediction per clip."""
    frame_indices_by_clip = defaultdict(list)
    for frame_position, sample in enumerate(samples):
        frame_indices_by_clip[sample["clip_id"]].append(frame_position)

    clip_ids = []
    clip_true_labels = []
    clip_probabilities = []
    clip_frame_positions = []

    for clip_id, frame_positions in frame_indices_by_clip.items():
        labels_for_clip = set(frame_labels[frame_positions].tolist())
        if len(labels_for_clip) != 1:
            raise ValueError(f"Clip contains conflicting frame labels: {clip_id}")

        clip_ids.append(clip_id)
        clip_true_labels.append(next(iter(labels_for_clip)))
        clip_probabilities.append(frame_probabilities[frame_positions].mean(axis=0))
        clip_frame_positions.append(frame_positions)

    clip_probabilities = np.asarray(clip_probabilities)
    clip_predictions = clip_probabilities.argmax(axis=1)
    return (
        clip_ids,
        np.asarray(clip_true_labels),
        clip_predictions,
        clip_probabilities,
        clip_frame_positions,
    )


frame_true, frame_pred, frame_probabilities = get_frame_predictions(
    model,
    test_loader,
    DEVICE,
)
(
    test_clip_ids,
    clip_true,
    clip_pred,
    clip_probabilities,
    clip_frame_positions,
) = aggregate_clip_predictions(test_samples, frame_true, frame_probabilities)

# Clip-level metrics are primary because inference returns one result per clip.
test_accuracy = float((clip_true == clip_pred).mean())
test_balanced_accuracy = balanced_accuracy_score(clip_true, clip_pred)
test_f1_macro = f1_score(clip_true, clip_pred, average="macro")
test_f1_weighted = f1_score(clip_true, clip_pred, average="weighted")
test_f1_per_class = f1_score(
    clip_true,
    clip_pred,
    labels=np.arange(len(CLASS_NAMES)),
    average=None,
    zero_division=0,
)

print("=" * 72)
print("HELD-OUT TEST RESULTS — CLIP LEVEL")
print("=" * 72)
print(f"Accuracy:          {test_accuracy:.4f}")
print(f"Balanced accuracy: {test_balanced_accuracy:.4f}")
print(f"F1 macro:          {test_f1_macro:.4f}")
print(f"F1 weighted:       {test_f1_weighted:.4f}")
print(f"Test clips:        {len(clip_true)}")
print(f"Test frames:       {len(frame_true)}")

print("\nF1 per class:")
for class_name, class_f1 in zip(CLASS_NAMES, test_f1_per_class):
    print(f"  {class_name:<20s} {class_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    clip_true,
    clip_pred,
    labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

In [ ]:
# Use VideoModule's shared renderer for count and normalized confusion matrices.
clip_confusion = confusion_matrix(
    clip_true,
    clip_pred,
    labels=np.arange(len(CLASS_NAMES)),
)
confusion_count_path = CFG.output_dir / "clip_confusion_matrix_counts.png"
confusion_normalized_path = CFG.output_dir / "clip_confusion_matrix_normalized.png"

render_confusion_matrix(
    clip_confusion,
    CLASS_NAMES,
    confusion_count_path,
    title="Held-out clip confusion matrix — counts",
)
render_confusion_matrix(
    clip_confusion,
    CLASS_NAMES,
    confusion_normalized_path,
    title="Held-out clip confusion matrix — recall",
    normalise=True,
)

display(DisplayImage(filename=str(confusion_count_path)))
display(DisplayImage(filename=str(confusion_normalized_path)))

In [ ]:
# One-vs-rest ROC and precision-recall curves at clip level.
clip_true_binary = label_binarize(
    clip_true,
    classes=np.arange(len(CLASS_NAMES)),
)
palette = sns.color_palette("husl", len(CLASS_NAMES))

figure, (roc_axis, pr_axis) = plt.subplots(1, 2, figsize=(16, 7))
figure.suptitle("Held-out clip performance curves", fontsize=16, fontweight="bold")

roc_auc_by_class = {}
average_precision_by_class = {}
for class_code, class_name in enumerate(CLASS_NAMES):
    false_positive_rate, true_positive_rate, _ = roc_curve(
        clip_true_binary[:, class_code],
        clip_probabilities[:, class_code],
    )
    class_roc_auc = auc(false_positive_rate, true_positive_rate)
    roc_auc_by_class[class_name] = class_roc_auc
    roc_axis.plot(
        false_positive_rate,
        true_positive_rate,
        color=palette[class_code],
        linewidth=2,
        label=f"{class_name} (AUC={class_roc_auc:.3f})",
    )

    precision, recall, _ = precision_recall_curve(
        clip_true_binary[:, class_code],
        clip_probabilities[:, class_code],
    )
    class_average_precision = average_precision_score(
        clip_true_binary[:, class_code],
        clip_probabilities[:, class_code],
    )
    average_precision_by_class[class_name] = class_average_precision
    pr_axis.plot(
        recall,
        precision,
        color=palette[class_code],
        linewidth=2,
        label=f"{class_name} (AP={class_average_precision:.3f})",
    )

macro_roc_auc = float(np.mean(list(roc_auc_by_class.values())))
macro_average_precision = float(np.mean(list(average_precision_by_class.values())))

roc_axis.plot([0, 1], [0, 1], "k--", alpha=0.4)
roc_axis.set_title(f"ROC curves — macro AUC {macro_roc_auc:.3f}")
roc_axis.set_xlabel("False positive rate")
roc_axis.set_ylabel("True positive rate")
roc_axis.legend()
roc_axis.grid(alpha=0.3)

pr_axis.set_title(f"Precision-recall curves — macro AP {macro_average_precision:.3f}")
pr_axis.set_xlabel("Recall")
pr_axis.set_ylabel("Precision")
pr_axis.legend()
pr_axis.grid(alpha=0.3)

curve_path = CFG.output_dir / "clip_roc_pr_curves.png"
plt.tight_layout()
figure.savefig(curve_path, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

In [ ]:
# Show the strongest predicted-class frame for each misclassified clip.
misclassified_clip_indices = np.where(clip_true != clip_pred)[0]
print(
    f"Misclassified clips: {len(misclassified_clip_indices)} / "
    f"{len(clip_true)}"
)

if len(misclassified_clip_indices) > 0:
    number_to_show = min(10, len(misclassified_clip_indices))
    columns = min(5, number_to_show)
    rows = (number_to_show + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).reshape(rows, columns)
    figure.suptitle("Misclassified clips — strongest predicted frame")

    for plot_position, clip_position in enumerate(
        misclassified_clip_indices[:number_to_show]
    ):
        predicted_class = clip_pred[clip_position]
        candidate_positions = clip_frame_positions[clip_position]
        strongest_frame_position = max(
            candidate_positions,
            key=lambda position: frame_probabilities[position, predicted_class],
        )
        sample = test_samples[strongest_frame_position]
        confidence = clip_probabilities[clip_position, predicted_class]

        row, column = divmod(plot_position, columns)
        axes[row, column].imshow(
            PILImage.open(sample["frame_path"]).convert("RGB")
        )
        axes[row, column].set_title(
            f"True: {CLASS_NAMES[clip_true[clip_position]]}\n"
            f"Pred: {CLASS_NAMES[predicted_class]} ({confidence:.1%})",
            color="red",
            fontsize=9,
        )
        axes[row, column].axis("off")

    for unused_position in range(number_to_show, rows * columns):
        row, column = divmod(unused_position, columns)
        axes[row, column].axis("off")

    misclassified_path = CFG.output_dir / "misclassified_clips.png"
    plt.tight_layout()
    figure.savefig(misclassified_path, bbox_inches="tight")
    plt.show()
else:
    print("No held-out clips were misclassified")

In [ ]:
# Confidence distribution for correct and incorrect clip predictions.
clip_confidence = clip_probabilities.max(axis=1)
correct_mask = clip_true == clip_pred

figure, axis = plt.subplots(figsize=(10, 5))
axis.hist(
    clip_confidence[correct_mask],
    bins=15,
    alpha=0.65,
    label=f"Correct ({correct_mask.sum()})",
    color="green",
)
if (~correct_mask).any():
    axis.hist(
        clip_confidence[~correct_mask],
        bins=15,
        alpha=0.65,
        label=f"Incorrect ({(~correct_mask).sum()})",
        color="red",
    )
axis.set_xlabel("Mean clip prediction confidence")
axis.set_ylabel("Clips")
axis.set_title("Clip confidence: correct versus incorrect")
axis.legend()
axis.grid(alpha=0.3)

confidence_path = CFG.output_dir / "clip_confidence_distribution.png"
plt.tight_layout()
figure.savefig(confidence_path, bbox_inches="tight")
plt.show()

---

## Chapter 9: Run optional video inference

This section is not used for CNN training. It decodes a supplied video with aspect ratio preserved, rotates camera footage dated from 21 June 2026 onward by 180?, samples at 60 FPS, and applies the same 0.5-second class-specific temporal rules used by the S3 notebook.

`fallen_in_view` uses consecutive-frame or supported-peak evidence, `fallen_before_entry` uses persistent median evidence, and safe classes require stable mean evidence. If no rule passes, the result remains `uncertain`.


In [ ]:
def infer_rotation(video_path, rotate_180):
    """Use an explicit override, or infer historical rotation from the filename."""
    if rotate_180 is not None:
        return bool(rotate_180)

    timestamp_match = VIDEO_TIMESTAMP_PATTERN.search(Path(video_path).name)
    if timestamp_match is None:
        print("No timestamp in filename; assuming the clip is already upright")
        return False

    video_timestamp = pd.to_datetime(
        timestamp_match.group(1),
        format="%Y-%m-%d_%H-%M-%S_%f",
    )
    return video_timestamp >= ROTATE_FROM_TIMESTAMP_UTC


@torch.no_grad()
def predict_clip(
    model,
    video_path,
    transform=val_test_transforms,
    device=DEVICE,
    sample_fps=CFG.inference_sample_fps,
    rotate_180=None,
):
    """Classify a clip using whole-clip rolling temporal windows."""
    video_path = Path(video_path)
    frames, fps = decode_clip_rgb_preserve_aspect_with_fps(
        video_path,
        maximum_side=256,
    )
    frames = apply_video_orientation(
        frames,
        infer_rotation(video_path, rotate_180),
    )
    selected_indices = select_inference_frame_indices(
        len(frames),
        fps,
        sample_fps,
    )

    frame_probability_batches = []
    for batch_start in range(0, len(selected_indices), CFG.batch_size):
        batch_indices = selected_indices[
            batch_start : batch_start + CFG.batch_size
        ]
        batch_tensors = [
            transform(PILImage.fromarray(frames[frame_index]))
            for frame_index in batch_indices
        ]
        image_batch = torch.stack(batch_tensors).to(device)

        with autocast_context(device):
            logits = model(image_batch)
        frame_probability_batches.append(
            torch.softmax(logits, dim=1).cpu().numpy()
        )

    frame_probabilities = np.concatenate(
        frame_probability_batches,
        axis=0,
    )
    temporal_result = calculate_temporal_clip_decision(
        frame_probabilities=frame_probabilities,
        frame_indices=selected_indices,
        source_fps=fps,
        class_names=CLASS_NAMES,
        rolling_window_seconds=CFG.rolling_window_seconds,
        fallen_threshold=CFG.fallen_threshold,
        falling_threshold=CFG.falling_threshold,
        normal_threshold=CFG.normal_threshold,
        fault_activity_threshold=CFG.fault_activity_threshold,
        falling_peak_threshold=CFG.falling_peak_threshold,
        persistent_seconds=CFG.persistent_seconds,
        min_falling_seconds=CFG.min_falling_seconds,
        peak_support_seconds=CFG.peak_support_seconds,
        normal_clear_seconds=CFG.normal_clear_seconds,
        fallen_clear_seconds=CFG.fallen_clear_seconds,
    )
    preview_index = selected_indices[temporal_result["preview_position"]]

    return {
        "path": str(video_path),
        "prediction": temporal_result["prediction"],
        "confidence": temporal_result["confidence"],
        # Keep this key for existing CSV/S3 consumers. Values are temporal scores.
        "probabilities": temporal_result["temporal_scores"],
        "temporal_scores": temporal_result["temporal_scores"],
        "source_frame_count": len(frames),
        "selected_frame_count": len(selected_indices),
        "selected_frame_indices": selected_indices,
        "source_fps": float(fps),
        "effective_sample_fps": temporal_result["effective_sample_fps"],
        "strongest_entry_window": temporal_result["strongest_entry_window"],
        "strongest_fall_window": temporal_result["strongest_fall_window"],
        "preview_frame": frames[preview_index],
        "preview_frame_index": int(preview_index),
        "preview_frame_time_seconds": float(preview_index / fps),
    }


print("Clip inference function defined")
print('Usage: result = predict_clip(model, Path("new_clip.ts"))')
print("Default: every frame. Use sample_fps=10.0 for consistent 10 FPS inference.")
print("Use rotate_180=True or False to override filename-based rotation")


In [ ]:
# Demonstrate the frame classifier on retained held-out original images.
demo_samples = [
    sample
    for sample in test_samples
    if sample["image_kind"] == "original"
][:3]
if not demo_samples:
    raise ValueError("No held-out original frames are available for the demo")

model.eval()
demo_probabilities = []
with torch.no_grad():
    for sample in demo_samples:
        image = PILImage.open(sample["frame_path"]).convert("RGB")
        image_tensor = val_test_transforms(image).unsqueeze(0).to(DEVICE)
        with autocast_context(DEVICE):
            logits = model(image_tensor)
        probabilities = torch.softmax(logits.float(), dim=1)[0].cpu().numpy()
        demo_probabilities.append(probabilities)

figure, axes = plt.subplots(1, len(demo_samples), figsize=(5 * len(demo_samples), 4))
axes = np.atleast_1d(axes)
figure.suptitle("Held-out frame inference demo")

for axis, sample, probabilities in zip(axes, demo_samples, demo_probabilities):
    true_label = sample["class_name"]
    predicted_index = int(np.argmax(probabilities))
    predicted_label = CLASS_NAMES[predicted_index]
    confidence = float(probabilities[predicted_index])
    prediction_is_correct = true_label == predicted_label

    image = PILImage.open(sample["frame_path"]).convert("RGB")
    axis.imshow(image)
    axis.set_title(
        f"True: {true_label}\n"
        f"Pred: {predicted_label} ({confidence:.1%})",
        color="green" if prediction_is_correct else "red",
    )
    axis.axis("off")

frame_demo_path = CFG.output_dir / "frame_inference_demo.png"
plt.tight_layout()
figure.savefig(frame_demo_path, bbox_inches="tight")
plt.show()
print(f"Saved: {frame_demo_path}")


---

## Chapter 10: Save the model handoff artefacts

In [ ]:
# Calculate checksums so S3 cannot silently load mismatched artefacts.
def calculate_file_sha256(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Save enough metadata to reproduce and validate the trained model.
metadata = {
    "model": "resnet50",
    "classifier_head": f"Dropout-Linear(2048,512)-ReLU-Dropout-Linear(512,{len(CLASS_NAMES)})",
    "class_names": CLASS_NAMES,
    "img_size": CFG.img_size,
    "training_frame_manifest": str(CFG.frame_manifest_path.resolve()),
    "frame_dataset_root": str(CFG.frame_dataset_root.resolve()),
    "training_dataset_policy": "self-contained frame dataset; video-reference and exact-content grouping; dates ignored; exact 50/50 original and ORB; balanced validation and test video counts",
    "training_balance_strategy": (
        "WeightedRandomSampler with unweighted CrossEntropyLoss"
    ),
    "leakage_audit": leakage_audit_summary,
    "inference_sample_fps": CFG.inference_sample_fps,
    "inference_aggregation_version": INFERENCE_AGGREGATION_VERSION,
    "rolling_window_seconds": CFG.rolling_window_seconds,
    "normal_threshold": CFG.normal_threshold,
    "fault_activity_threshold": CFG.fault_activity_threshold,
    "falling_threshold": CFG.falling_threshold,
    "falling_peak_threshold": CFG.falling_peak_threshold,
    "fallen_threshold": CFG.fallen_threshold,
    "persistent_seconds": CFG.persistent_seconds,
    "min_falling_seconds": CFG.min_falling_seconds,
    "peak_support_seconds": CFG.peak_support_seconds,
    "normal_clear_seconds": CFG.normal_clear_seconds,
    "fallen_clear_seconds": CFG.fallen_clear_seconds,
    "live_inference_policy": "FPS-aware class-specific temporal rules with uncertain state",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "handoff_schema_version": MODEL_HANDOFF_SCHEMA_VERSION,
    "image_preprocessing_version": IMAGE_PREPROCESSING_VERSION,
    "image_preprocessing_policy": "preserve aspect ratio and pad to square with ImageNet mean RGB",
    "letterbox_fill_rgb": list(LETTERBOX_FILL),
    "rotate_from_timestamp_utc": str(ROTATE_FROM_TIMESTAMP_UTC),
    "s3_rotation_policy": "rotate 180 degrees at or after the UTC cutoff",
    "training_orientation_policy": (
        "normal source frames rotated 180 degrees; no_bottle and fallen classes "
        "retain their checked orientation; all ORB pairs use shared references"
    ),
    "training_manifest_sha256": calculate_file_sha256(CFG.frame_manifest_path),
    "model_checkpoint_sha256": calculate_file_sha256(CFG.model_save_path),
    "test_accuracy_clip": float(test_accuracy),
    "test_balanced_accuracy_clip": float(test_balanced_accuracy),
    "test_f1_macro_clip": float(test_f1_macro),
    "test_f1_weighted_clip": float(test_f1_weighted),
    "test_f1_per_class_clip": {
        class_name: float(class_f1)
        for class_name, class_f1 in zip(CLASS_NAMES, test_f1_per_class)
    },
    "roc_auc_per_class_clip": roc_auc_by_class,
    "roc_auc_macro_clip": macro_roc_auc,
    "average_precision_per_class_clip": average_precision_by_class,
    "average_precision_macro_clip": macro_average_precision,
    "train_frames": len(train_dataset),
    "validation_frames": len(val_dataset),
    "test_frames": len(test_dataset),
    "train_videos": len(train_clip_set),
    "validation_videos": len(val_clip_set),
    "test_videos": len(test_clip_set),
    "train_pairs": len(train_pair_set),
    "validation_pairs": len(val_pair_set),
    "test_pairs": len(test_pair_set),
    "epochs_head": CFG.epochs_head,
    "epochs_finetune": CFG.epochs_finetune,
    "checkpoint_selection_metric": "validation_clip_f1_macro",
    "best_validation_clip_f1_macro": float(max(history.val_clip_f1_macro)),
    "best_validation_loss": float(min(history.val_loss)),
    "best_validation_accuracy": float(max(history.val_accuracy)),
}

metadata_path = CFG.output_dir / "model_metadata.json"
with metadata_path.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

# Record every original/ORB pair and its split for reproducibility.
split_manifest = [
    {
        "clip_id": pair_record["clip_id"],
        "source_image_path": pair_record["source_image_path"],
        "class_name": pair_record["class_name"],
        "split": pair_record["split"],
        "original_image_path": pair_record["original_image_path"],
        "orb_image_path": pair_record["orb_image_path"],
    }
    for pair_record in pair_records
]
split_manifest_path = CFG.output_dir / "split_manifest.json"
with split_manifest_path.open("w", encoding="utf-8") as manifest_file:
    json.dump(split_manifest, manifest_file, indent=2)

print(f"Model weights:  {CFG.model_save_path}")
print(f"Model metadata: {metadata_path}")
print(f"Split manifest: {split_manifest_path}")
print("\nReload with:")
print("  model = build_model()")
print(f"  model.load_state_dict(torch.load(r'{CFG.model_save_path}', weights_only=True))")
print("  model = model.to(DEVICE).eval()")


---

## Pipeline summary

### Four-class training and inference contract

1. Load `stoppage_detection_and_classification/cnn_classifier/data/training_frames/training_frames_50_manifest.csv` and validate every retained image path.
2. Require one original and one ORB-transformed copy for every checked source frame.
3. Preserve complete video references and exact-content-linked references within one split, while ignoring dates.
4. Balance validation and test by video count for all four classes.
5. Preserve image aspect ratio with neutral letterbox padding in training and inference.
6. Train the ResNet50 head, then fine-tune `layer3` and `layer4`.
7. Save the epoch with the best validation clip-level macro F1, using validation loss as the tie-break.
8. Evaluate held-out frame and clip behaviour, including per-class F1, confusion matrices, ROC and precision-recall curves.
9. Save a schema-3 metadata handoff containing the manifest/model hashes, four-class mapping, preprocessing contract and 60 FPS temporal thresholds required by S3 inference.

Because preprocessing changed from stretched squares to aspect-preserving letterboxing, the existing schema-2 checkpoint must be retrained before S3 inference.
